# Ledger Construction

In [1]:
from pathlib import Path
import contextlib, importlib.util, io, json
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
def load_script(filename):
    spec = importlib.util.spec_from_file_location(filename.replace(".py", ""), ROOT / "scripts" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
emit10 = load_script("10_emit_skyportal_source_facts.py")
fetch11 = load_script("11_fetch_source_detail.py")
emit13 = load_script("13_emit_gcn_facts.py")
LEDGER = ROOT / "data" / "ledger"
skyportal_control = pd.concat([pd.read_parquet(path) for path in sorted((LEDGER / "facts" / "source_system=skyportal").glob("year=*/part-*.parquet"))], ignore_index=True)
gcn_control = pd.concat([pd.read_parquet(path) for path in sorted((LEDGER / "facts" / "source_system=gcn").glob("year=*/part-*.parquet"))], ignore_index=True)
events_control = pd.read_parquet(LEDGER / "events" / "events.parquet")
direct_map_control = pd.read_parquet(LEDGER / "event_container_map" / "map.parquet")
gcn_map_control = pd.read_parquet(LEDGER / "event_container_map" / "gcn_map.parquet")
print(f"Loaded controls: {len(skyportal_control):,} SkyPortal facts; {len(gcn_control):,} GCN facts; {len(events_control):,} events; {len(direct_map_control):,} direct-map rows; {len(gcn_map_control):,} GCN-map rows")
assert (len(skyportal_control), len(events_control), len(direct_map_control), len(gcn_map_control)) == (1776, 800, 800, 2335)
ARCHIVE_DIR = ROOT / "data" / "raw" / "gcn" / "circulars" / "archive_json" / "20260720_093324" / "extracted" / "archive.json"
DETAIL_DIR = ROOT / "data" / "raw" / "skyportal" / "source_detail_20260724"

Loaded controls: 1,776 SkyPortal facts; 101,072 GCN facts; 800 events; 800 direct-map rows; 2,335 GCN-map rows


## 1. The Question

The ledger exists, but is it reproducible from the frozen raw captures? This notebook regenerates facts by invoking the production emitters in memory. The ledger is read only as a control. Every reported fact count places regenerated output beside that control.

## 2. The Construction Recipe

Four scripts connect the frozen captures to ledger facts and container maps. The table records their role and the current on-disk control size.

In [2]:
capture_records, capture_times = emit10.load_capture()
unique_sources, source_profiles, source_origins = emit10.deduplicate(capture_records)
detail_counts = {}
for path in sorted(DETAIL_DIR.glob("*/*.json")):
    wrapper = json.loads(path.read_text())
    records = fetch11.extract_records(path.stem, wrapper.get("payload"))
    detail_counts[path.stem] = detail_counts.get(path.stem, 0) + (len(records) if isinstance(records, list) else 0)
circulars_by_year = {}
for path in sorted(ARCHIVE_DIR.glob("*.json")):
    raw = json.loads(path.read_text())
    created = pd.to_datetime(raw["createdOn"], unit="ms", utc=True)
    if created.year < 2023 or len(str(raw.get("body") or "").strip()) < 200:
        continue
    circulars_by_year.setdefault(created.year, []).append({"circular_id":int(raw["circularId"]),"subject":str(raw.get("subject") or ""),"body":str(raw["body"]),"event_id":raw.get("eventId"),"created_on":created.isoformat(),"submitter":raw.get("submitter")})
for rows in circulars_by_year.values():
    rows.sort(key=lambda row: row["circular_id"])
recipe_table = pd.DataFrame([
    {"script":"10","reads":"982 frozen listing rows","emits":"SkyPortal facts; events; direct map","rows on disk":"1,776; 800; 800"},
    {"script":"11","reads":"800 source ids + SkyPortal API","emits":"raw source-detail collections","rows on disk":f"{sum(detail_counts.values()):,} records"},
    {"script":"13","reads":f"{sum(map(len, circulars_by_year.values())):,} eligible GCN circulars","emits":"GCN facts","rows on disk":f"{len(gcn_control):,}"},
    {"script":"14","reads":"800 events + event selector","emits":"GCN event-container map","rows on disk":f"{len(gcn_map_control):,}"},
])
print(recipe_table.to_string(index=False))

script                          reads                               emits    rows on disk
    10        982 frozen listing rows SkyPortal facts; events; direct map 1,776; 800; 800
    11 800 source ids + SkyPortal API       raw source-detail collections  13,278 records
    13  12,012 eligible GCN circulars                           GCN facts         101,072
    14    800 events + event selector             GCN event-container map           2,335


## 3. Full Regeneration, SkyPortal Side

All source-level facts are regenerated from the frozen listing through script 10. Comparison uses `fact_id`, then checks `t_known`, `t_known_method`, and `value_raw` for common facts.

In [3]:
COMPARE_FIELDS = ["t_known", "t_known_method", "value_raw"]
def compare_fact_frames(regenerated, control):
    regenerated = regenerated.set_index("fact_id", drop=False); control = control.set_index("fact_id", drop=False)
    regenerated_ids, control_ids = set(regenerated.index), set(control.index); common = sorted(regenerated_ids & control_ids)
    def values(frame, fact_id):
        if fact_id not in frame.index: return None
        row = frame.loc[fact_id]; known = pd.to_datetime(row["t_known"], utc=True)
        return {"t_known":None if pd.isna(known) else known.isoformat(),"t_known_method":None if pd.isna(row["t_known_method"]) else str(row["t_known_method"]),"value_raw":None if pd.isna(row["value_raw"]) else str(row["value_raw"])}
    field_differences = [fact_id for fact_id in common if values(regenerated, fact_id) != values(control, fact_id)]
    mismatch_ids = sorted((regenerated_ids ^ control_ids) | set(field_differences))
    metrics = {"rows regenerated":len(regenerated),"rows in ledger":len(control),"fact_ids matching":len(common),"only regeneration":len(regenerated_ids-control_ids),"only ledger":len(control_ids-regenerated_ids),"field differences":len(field_differences)}
    rows = [{"row type":"summary", **metrics, "fact_id":"", "regenerated":"", "ledger":""}]
    for fact_id in mismatch_ids[:3]:
        rows.append({"row type":"mismatch sample","fact_id":fact_id,"regenerated":json.dumps(values(regenerated,fact_id),sort_keys=True),"ledger":json.dumps(values(control,fact_id),sort_keys=True)})
    return pd.DataFrame(rows), metrics
regenerated_skyportal_rows, parse_failures = emit10.build_facts(unique_sources, source_origins, capture_times)
regenerated_skyportal = pd.DataFrame(regenerated_skyportal_rows)
skyportal_table, skyportal_metrics = compare_fact_frames(regenerated_skyportal, skyportal_control)
print(skyportal_table.fillna("").to_string(index=False))

row type  rows regenerated  rows in ledger  fact_ids matching  only regeneration  only ledger  field differences fact_id regenerated ledger
 summary              1776            1776               1776                  0            0                  0                           


## 4. Sampled Regeneration, GCN Side

The 50-circular sample is allocated proportionally by year using largest remainders, then takes the lowest circular ids deterministically. Script 13 performs its normal extraction and deduplication while its disk writers are replaced by an in-memory sink.

In [4]:
eligible_total = sum(map(len, circulars_by_year.values()))
quotas = {year:int(len(rows) * 50 / eligible_total) for year, rows in circulars_by_year.items()}
remainders = sorted(circulars_by_year, key=lambda year:(-(len(circulars_by_year[year]) * 50 / eligible_total - quotas[year]), year))
for year in remainders[:50 - sum(quotas.values())]:
    quotas[year] += 1
sample = [row for year in sorted(circulars_by_year) for row in circulars_by_year[year][:quotas[year]]]
regenerated_gcn_rows = []
emit13.iter_real_circulars = lambda min_year=2023: iter(sample)
emit13.write_year = lambda year, rows: regenerated_gcn_rows.extend(rows)
for function_name in ["write_audit", "write_sample", "write_fix_audit", "print_fix_summary"]:
    setattr(emit13, function_name, lambda *args, **kwargs: None)
with contextlib.redirect_stdout(io.StringIO()):
    emit13.main()
sampled_ids = {str(row["circular_id"]) for row in sample}
sampled_control = gcn_control[gcn_control["container_id"].astype(str).isin(sampled_ids)].copy()
gcn_table, gcn_metrics = compare_fact_frames(pd.DataFrame(regenerated_gcn_rows), sampled_control)
composition = pd.DataFrame([{"row type":"sample year","year":year,"eligible circulars":len(circulars_by_year[year]),"sampled circulars":quotas[year]} for year in sorted(quotas)])
section4_table = pd.concat([composition, gcn_table], ignore_index=True, sort=False)
print(section4_table.fillna("").to_string(index=False))

   row type    year eligible circulars sampled circulars rows regenerated rows in ledger fact_ids matching only regeneration only ledger field differences fact_id regenerated ledger
sample year  2023.0             2308.0               9.0                                                                                                                             
sample year  2024.0             3284.0              14.0                                                                                                                             
sample year  2025.0             4528.0              19.0                                                                                                                             
sample year  2026.0             1892.0               8.0                                                                                                                             
    summary                                                         292.0          292.0  

## 5. What This Means

The SkyPortal side is fully reproducible: all 1,776 regenerated facts match the ledger by identifier and checked fields. The GCN test regenerates 292 facts from 50 circulars, and all match their ledger controls. Those circulars cover 0.4163% of the 12,012 eligible GCN corpus, while their facts cover 0.2889% of the 101,072 GCN ledger facts. No identifier or checked-field mismatch was found.

Full GCN regeneration remains unverified because this notebook deliberately uses a year-stratified sample.

In [5]:
check_rows = []
for side, metrics in [("skyportal", skyportal_metrics), ("gcn_sample", gcn_metrics)]:
    expectations = {"rows regenerated":metrics["rows in ledger"],"rows in ledger":metrics["rows in ledger"],"fact_ids matching":metrics["rows in ledger"],"only regeneration":0,"only ledger":0,"field differences":0}
    for check, expected in expectations.items():
        observed = metrics[check]
        check_rows.append({"side":side,"check":check,"expected":expected,"observed":observed,"status":"PASS" if expected == observed else "MISMATCH"})
check_rows.append({"side":"gcn_sample","check":"sampled circulars","expected":50,"observed":len(sample),"status":"PASS" if len(sample) == 50 else "MISMATCH"})
regeneration_check = pd.DataFrame(check_rows, columns=["side", "check", "expected", "observed", "status"])
EXPORT_PATH = ROOT / "notebooks" / "evidence" / "06_regeneration_check.csv"
regeneration_check.to_csv(EXPORT_PATH, index=False)
print(f"Evidence shape: {regeneration_check.shape}")
print(regeneration_check.to_string(index=False))

Evidence shape: (13, 5)
      side             check  expected  observed status
 skyportal  rows regenerated      1776      1776   PASS
 skyportal    rows in ledger      1776      1776   PASS
 skyportal fact_ids matching      1776      1776   PASS
 skyportal only regeneration         0         0   PASS
 skyportal       only ledger         0         0   PASS
 skyportal field differences         0         0   PASS
gcn_sample  rows regenerated       292       292   PASS
gcn_sample    rows in ledger       292       292   PASS
gcn_sample fact_ids matching       292       292   PASS
gcn_sample only regeneration         0         0   PASS
gcn_sample       only ledger         0         0   PASS
gcn_sample field differences         0         0   PASS
gcn_sample sampled circulars        50        50   PASS
